In [ ]:
!pip install -q scikit-learn

In [18]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_from_disk, concatenate_datasets, Audio
import glob
from datasets import disable_caching
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    roc_curve,
    auc,
    roc_auc_score,
)

from torchvision.models import resnet18

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings

/content/drive/MyDrive/Speaker-Recognition-using-ResNet-Embeddings


## Load Data and DataLoader (dev and test set)

In [6]:
disable_caching()

# loading data

chunk_paths = sorted(glob.glob("speaker_chunks/chunk_*"))

chunks = []
for path in chunk_paths:
    ds = load_from_disk(path)
    ds = ds.cast_column("audio_path", Audio(sampling_rate=16000))
    chunks.append(ds)

full_dataset = concatenate_datasets(chunks)

In [7]:
split = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]

temp = split["test"].train_test_split(test_size=0.5, seed=42)
dev_dataset = temp["train"]
test_dataset = temp["test"]

In [8]:
from dataset import Dataset_Builder

config = {"shortest_duration": 4.0}
dev_builder = Dataset_Builder(dev_dataset, **config)
dev_builder.filter()
dev_builder.preprocess()

test_builder = Dataset_Builder(test_dataset, **config)
test_builder.filter()
test_builder.preprocess()

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2402 [00:00<?, ? examples/s]

Duration of cropped log-mel spectograms (input):  4.0  seconds


Filter:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2407 [00:00<?, ? examples/s]

In [9]:
dev_dataset = dev_builder.get_dataset()
test_dataset = test_builder.get_dataset()

In [10]:
def collate_fn(batch):
    log_mels = torch.stack([x["log_mel"]for x in batch])
    labels = torch.tensor([x["label"] for x in batch],dtype=torch.long)

    return {
        "log_mel": log_mels,
        "labels": labels
        }

In [13]:
dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

## Build Model and Load checkpoint

In [17]:
NUM_CLASSES = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "checkpoint/"

model = resnet18(weights=None)
model.fc = torch.nn.Linear(in_features=512, out_features=NUM_CLASSES, bias=True)

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

NameError: name 'ResNet18' is not defined

## Accuracy Evaluation (for dev set)

In [ ]:
all_labels = []
all_probs = []
all_preds = []

correct  = 0
total = 0

with torch.no_grad():
  for indx, (log_mels, labels) in tqdm(enumerate(dev_loader), total=len(dev_loader)):
      log_mels = log_mels.to(DEVICE)
      labels = labels.to(DEVICE)
      logits = model(log_mels)
      probs = torch.softmax(logits, dim=1)
      preds = torch.argmax(probs, dim=1)

      correct += torch.sum(preds == labels).item()
      total += len(labels)

      all_labels.extend(labels.cpu().numpy())
      all_probs.append(probs.cpu().numpy())
      all_preds.extend(preds.cpu().numpy())

accuracy = correct / total
print(f"Accuracy = {accuracy:4f}")

## Confusion matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds, normalize="true")

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(xticks_rotation=90, cmap="Blues", colorbar=True)
plt.title("Confusion Matrix")
plt.show()

## ROC Evaluation

In [19]:
all_probs = np.concatenate(all_probs, axis=0)
all_labels = np.array(all_labels)

y_true = label_binarize(all_labels, classes=np.arange(NUM_CLASSES))

fpr = {}
tpr = {}
roc_auc = {}

for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_true[:, i], all_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_true.ravel(), all_probs.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

roc_auc["macro"] = roc_auc_score(y_true, all_probs, average="macro", multi_class="ovr")

print(f"Micro-average ROC AUC: {roc_auc['micro']:.3f}")
print(f"Macro-average ROC AUC: {roc_auc['macro']:.3f}")


NameError: name 'all_probs' is not defined

## Plot out ROC Curve

In [ ]:
plt.figure(figsize=(6,6))

plt.plot(
    fpr["micro"],
    tpr["micro"],
    label=f"Micro-average ROC (AUC = {roc_auc['micro']:.3f})"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Speaker Identification ROC Curve")
plt.legend()

plt.show()

In [ ]:
fig, axes = plt.subplots(5, 10, figsize=(18, 9))

axes = axes.ravel()

for i in range(NUM_CLASSES):

    axes[i].plot(fpr[i], tpr[i], linewidth=1.5, label=f"AUC = {roc_auc[i]:.3f}")
    axes[i].set_title(f"S{i}", fontsize=8)

    axes[i].set_xticks([])
    axes[i].set_yticks([])

    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1)

plt.tight_layout()
plt.show()